In [1]:
%pip install pyspark pandas numpy

import pandas as pd
import numpy as np
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark Learn 2").master("spark://localhost:7077").getOrCreate()

print(spark)


Note: you may need to restart the kernel to use updated packages.


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/09/08 11:59:40 WARN Utils: Your hostname, soc-5CG2243S91, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/09/08 11:59:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/08 11:59:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.sql import functions as sf
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, DateType

# Customer Data
customers_data = [
    (101, "Alice", "London"),
    (102, "Bob", "New York"),
    (103, "Charlie", "London"),
    (104, "David", "Tokyo")
]
customer_schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("name", StringType()),
    StructField("city", StringType())
])
customers_df = spark.createDataFrame(customers_data, customer_schema)

In [4]:

# Sales Data
sales_data = [
    (1, 101, "2025-01-15", 2, 150.0),
    (2, 102, "2025-01-16", 1, 200.0),
    (3, 101, "2025-01-17", 5, 50.0),
    (4, 103, "2025-02-01", 3, 75.0),
    (5, 102, "2025-02-02", 2, 250.0),
    (6, 101, "2025-02-03", 1, 175.0),
    (7, 104, "2025-03-10", 10, 30.0),
    (8, 103, "2025-03-12", 1, 120.0)
]
sales_schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("order_date", StringType()),
    StructField("quantity", IntegerType()),
    StructField("price_per_unit", FloatType())
])
sales_df = spark.createDataFrame(sales_data, sales_schema) \
    .withColumn("order_date", sf.to_date("order_date", format="yyyy-MM-dd")) \
    .withColumn("total_price", sf.round(sf.col("quantity") * sf.col("price_per_unit"), 2))

print("DataFrames created. Ready for tasks.")

DataFrames created. Ready for tasks.


In [5]:
print(sales_df.rdd.getNumPartitions())
sales_df.explain()

8
== Physical Plan ==
*(1) Project [order_id#10, customer_id#11, cast(gettimestamp(order_date#12, yyyy-MM-dd, TimestampType, try_to_date, Some(Etc/UTC), true) as date) AS order_date#15, quantity#13, price_per_unit#14, round((cast(quantity#13 as double) * cast(price_per_unit#14 as double)), 2) AS total_price#16]
+- *(1) Scan ExistingRDD[order_id#10,customer_id#11,order_date#12,quantity#13,price_per_unit#14]




In [6]:
sales_df.show()

+--------+-----------+----------+--------+--------------+-----------+
|order_id|customer_id|order_date|quantity|price_per_unit|total_price|
+--------+-----------+----------+--------+--------------+-----------+
|       1|        101|2025-01-15|       2|         150.0|      300.0|
|       2|        102|2025-01-16|       1|         200.0|      200.0|
|       3|        101|2025-01-17|       5|          50.0|      250.0|
|       4|        103|2025-02-01|       3|          75.0|      225.0|
|       5|        102|2025-02-02|       2|         250.0|      500.0|
|       6|        101|2025-02-03|       1|         175.0|      175.0|
|       7|        104|2025-03-10|      10|          30.0|      300.0|
|       8|        103|2025-03-12|       1|         120.0|      120.0|
+--------+-----------+----------+--------+--------------+-----------+



In [7]:
joined_df = sales_df.join(customers_df, on="customer_id", how="inner")
print(joined_df.rdd.getNumPartitions())
joined_df.show()

1


+-----------+--------+----------+--------+--------------+-----------+-------+--------+
|customer_id|order_id|order_date|quantity|price_per_unit|total_price|   name|    city|
+-----------+--------+----------+--------+--------------+-----------+-------+--------+
|        101|       1|2025-01-15|       2|         150.0|      300.0|  Alice|  London|
|        101|       3|2025-01-17|       5|          50.0|      250.0|  Alice|  London|
|        101|       6|2025-02-03|       1|         175.0|      175.0|  Alice|  London|
|        102|       5|2025-02-02|       2|         250.0|      500.0|    Bob|New York|
|        102|       2|2025-01-16|       1|         200.0|      200.0|    Bob|New York|
|        103|       4|2025-02-01|       3|          75.0|      225.0|Charlie|  London|
|        103|       8|2025-03-12|       1|         120.0|      120.0|Charlie|  London|
|        104|       7|2025-03-10|      10|          30.0|      300.0|  David|   Tokyo|
+-----------+--------+----------+--------+-

25/09/08 16:51:20 WARN StandaloneAppClient$ClientEndpoint: Connection to 6bfaa47937cb:7077 failed; waiting for master to reconnect...
25/09/08 16:51:20 WARN StandaloneSchedulerBackend: Disconnected from Spark cluster! Waiting for reconnection...
25/09/08 16:51:25 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.3: Remote RPC client disassociated. Likely due to containers exceeding thresholds, or network issues. Check driver logs for WARN messages.
25/09/08 16:51:26 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.2: Remote RPC client disassociated. Likely due to containers exceeding thresholds, or network issues. Check driver logs for WARN messages.


In [9]:
joined_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 2
   +- *(5) Project [customer_id#11, order_id#10, order_date#15, quantity#13, price_per_unit#14, total_price#16, name#1, city#2]
      +- *(5) SortMergeJoin [customer_id#11], [customer_id#0], Inner
         :- *(3) Sort [customer_id#11 ASC NULLS FIRST], false, 0
         :  +- AQEShuffleRead coalesced
         :     +- ShuffleQueryStage 0
         :        +- Exchange hashpartitioning(customer_id#11, 200), ENSURE_REQUIREMENTS, [plan_id=64]
         :           +- *(1) Project [order_id#10, customer_id#11, cast(gettimestamp(order_date#12, yyyy-MM-dd, TimestampType, try_to_date, Some(Etc/UTC), true) as date) AS order_date#15, quantity#13, price_per_unit#14, round((cast(quantity#13 as double) * cast(price_per_unit#14 as double)), 2) AS total_price#16]
         :              +- *(1) Filter isnotnull(customer_id#11)
         :                 +- *(1) Scan ExistingRDD[order_id#10,customer_id#11,o

In [6]:
revenue_by_city_df = joined_df.groupBy("city").agg(sf.sum("total_price").alias("total_revenue"))
revenue_by_city_df.show()

+--------+-------------+
|    city|total_revenue|
+--------+-------------+
|  London|       1070.0|
|   Tokyo|        300.0|
|New York|        700.0|
+--------+-------------+



In [7]:
from pyspark.sql.window import Window

windowSpec = Window.partitionBy("customer_id").orderBy("order_date")

running_total_df = sales_df.withColumn("running_total", sf.sum("total_price").over(window=windowSpec))
running_total_df.show()

+--------+-----------+----------+--------+--------------+-----------+-------------+
|order_id|customer_id|order_date|quantity|price_per_unit|total_price|running_total|
+--------+-----------+----------+--------+--------------+-----------+-------------+
|       1|        101|2025-01-15|       2|         150.0|      300.0|        300.0|
|       3|        101|2025-01-17|       5|          50.0|      250.0|        550.0|
|       6|        101|2025-02-03|       1|         175.0|      175.0|        725.0|
|       2|        102|2025-01-16|       1|         200.0|      200.0|        200.0|
|       5|        102|2025-02-02|       2|         250.0|      500.0|        700.0|
|       4|        103|2025-02-01|       3|          75.0|      225.0|        225.0|
|       8|        103|2025-03-12|       1|         120.0|      120.0|        345.0|
|       7|        104|2025-03-10|      10|          30.0|      300.0|        300.0|
+--------+-----------+----------+--------+--------------+-----------+-------

In [8]:
sales_df.groupBy("customer_id").agg(sf.sum("total_price").alias("total_spent")).show(vertical=True)

-RECORD 0------------
 customer_id | 102   
 total_spent | 700.0 
-RECORD 1------------
 customer_id | 103   
 total_spent | 345.0 
-RECORD 2------------
 customer_id | 101   
 total_spent | 725.0 
-RECORD 3------------
 customer_id | 104   
 total_spent | 300.0 



In [9]:
# Cache the DataFrame in memory
joined_df.cache()

# Run an action to trigger the join and caching
print(f"Count of joined rows: {joined_df.count()}")

# Run the same action again
print(f"Count of joined rows (again): {joined_df.count()}")

Count of joined rows: 8
Count of joined rows (again): 8


In [17]:
window = Window.partitionBy("customer_id").orderBy(sf.desc("total_price"))

ranked_df = sales_df.withColumn('rank', sf.rank().over(window))
ranked_df.orderBy("customer_id", "rank").show()

+--------+-----------+----------+--------+--------------+-----------+----+
|order_id|customer_id|order_date|quantity|price_per_unit|total_price|rank|
+--------+-----------+----------+--------+--------------+-----------+----+
|       1|        101|2025-01-15|       2|         150.0|      300.0|   1|
|       3|        101|2025-01-17|       5|          50.0|      250.0|   2|
|       6|        101|2025-02-03|       1|         175.0|      175.0|   3|
|       5|        102|2025-02-02|       2|         250.0|      500.0|   1|
|       2|        102|2025-01-16|       1|         200.0|      200.0|   2|
|       4|        103|2025-02-01|       3|          75.0|      225.0|   1|
|       8|        103|2025-03-12|       1|         120.0|      120.0|   2|
|       7|        104|2025-03-10|      10|          30.0|      300.0|   1|
+--------+-----------+----------+--------+--------------+-----------+----+



25/09/08 11:58:56 WARN StandaloneAppClient$ClientEndpoint: Connection to 0f6f697d43ae:7077 failed; waiting for master to reconnect...
25/09/08 11:58:56 WARN StandaloneSchedulerBackend: Disconnected from Spark cluster! Waiting for reconnection...
25/09/08 11:59:01 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.4: Remote RPC client disassociated. Likely due to containers exceeding thresholds, or network issues. Check driver logs for WARN messages.
25/09/08 11:59:01 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.2: Remote RPC client disassociated. Likely due to containers exceeding thresholds, or network issues. Check driver logs for WARN messages.
25/09/08 11:59:01 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_75_72!
25/09/08 11:59:01 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_75_195!
25/09/08 11:59:01 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_75_109!
25/09/08 11:59:01 WARN BlockManagerMasterEndpoint: No